### Cascaded Multilingual Speech-to-Speech (Unit 7)
The objective is to take the cascaded speech-to-speech translation Gradio demo from the first section in this Unit, and update it to translate to any non-English language. That is to say, the demo should take speech in language X, and translate it to speech in language Y, where the target language Y is not English. You should start by duplicating the template under your Hugging Face namespace. There’s no requirement to use a GPU accelerator device - the free CPU tier works just fine 🤗 However, you should ensure that the visibility of your demo is set to public. This is required such that your demo is accessible to us and can thus be checked for correctness.

In [ ]:
import numpy as np
import torch
import librosa
from IPython.display import Audio

from datasets import load_dataset

from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor, pipeline, AutoTokenizer
from transformers import VitsModel, VitsTokenizer

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# load speech translation checkpoint
asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=device)

# load text-to-speech checkpoint and speaker embeddings
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")

model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

ru_model = VitsModel.from_pretrained("facebook/mms-tts-rus")
ru_tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-rus")

model_deu = VitsModel.from_pretrained("facebook/mms-tts-deu")
tokenizer_deu = VitsTokenizer.from_pretrained("facebook/mms-tts-deu")

def translate(audio):
    #outputs = asr_pipe(audio, max_new_tokens=256, generate_kwargs={"task": "translate"})
    outputs = asr_pipe(audio, generate_kwargs={"task": "translate"})
    return outputs["text"]


def synthesise(text):
    inputs = processor(text=text, return_tensors="pt")
    speech = model.generate_speech(inputs["input_ids"].to(device), speaker_embeddings.to(device), vocoder=vocoder)
    return speech.cpu()


def synthesise_ru(text):    
    inputs = ru_tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]
    
    with torch.no_grad():
        speech = ru_model(input_ids.to('cpu'))
    return speech["waveform"].cpu()


def speech_to_speech_translation(audio):
    translated_text = translate(audio)
    synthesised_speech = synthesise(translated_text)
    #synthesised_speech = synthesise(translated_text)
    synthesised_speech = (synthesised_speech.numpy() * 32767).astype(np.int16)    
    return 16000, synthesised_speech

def speech_to_speech_translation_ru(audio):
    translated_text = translate(audio)
    #synthesised_speech = synthesise(translated_text)    
    synthesised_speech = synthesise_ru(translated_text)
    synthesised_speech = (synthesised_speech.numpy() * 32767).astype(np.int16)
    #synthesised_speech = synthesised_speech.numpy().astype(np.int16)
    return 16000, synthesised_speech

def synthesise_deu(text):            
    inputs = tokenizer_deu(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model_deu(input_ids.to('cpu'))

    speech = outputs["waveform"]
    return speech.cpu()


def speech_to_speech_translation_deu(audio):
    translated_text = translate(audio)
    #synthesised_speech = synthesise(translated_text)
    synthesised_speech = synthesise_deu(translated_text)
    synthesised_speech = (synthesised_speech.numpy() * 32767).astype(np.int16)    
    return 16000, synthesised_speech

Device set to use cuda:0


In [ ]:
audio, sample_rate = librosa.load("example.wav", sr=16000)

In [ ]:
sample_rate, translated_audio = speech_to_speech_translation(audio)

In [ ]:
translated_audio.shape

(191488,)

In [ ]:
Audio(translated_audio, rate=sample_rate)

In [ ]:
sample_rate, translated_audio_deu = speech_to_speech_translation_deu(audio)

In [ ]:
translated_audio_deu.shape

(1, 214784)

In [ ]:
Audio(translated_audio_deu[0], rate=sample_rate)

In [ ]:
sample_rate, translated_audio_ru = speech_to_speech_translation_ru(audio)

In [ ]:
translated_audio_ru.shape

(1, 53760)

In [ ]:
Audio(translated_audio_ru[0], rate=sample_rate)

In [ ]:
translated_audio

array([ -2,   6,   0, ..., -20, -15, -18], dtype=int16)

In [ ]:
translated_audio.max()

np.int16(10525)

In [ ]:
translated_audio.min()

np.int16(-12783)

In [ ]:
translated_audio_ru

array([[-20, -21, -11, ...,  -9,  16,  33]], dtype=int16)

In [ ]:
translated_audio_ru[0].max()

np.int16(11553)

In [ ]:
translated_audio_ru.min()

np.int16(-12778)

In [ ]:
from transformers import VitsModel, VitsTokenizer

model = VitsModel.from_pretrained("facebook/mms-tts-deu")
tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-deu")

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

C:\Users\Kseniia\anaconda3\envs\huggingface\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kseniia\.cache\huggingface\hub\models--facebook--mms-tts-deu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

#model = VitsModel.from_pretrained("facebook/mms-tts-rus")
#tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-rus")

#text = "some example text in the Russian language"
text = (
    "Ich bin Schnappi das kleine Krokodil, komm aus Ägypten das liegt direkt am Nil."
)

inputs = tokenizer(text, return_tensors="pt")

input_ids = inputs["input_ids"]

with torch.no_grad():
    outputs = model(input_ids)

speech = outputs["waveform"]

In [ ]:
outputs

VitsModelOutput(waveform=tensor([[ 0.0002,  0.0001,  0.0001,  ..., -0.0004, -0.0004, -0.0004]]), sequence_lengths=tensor([92416]), spectrogram=tensor([[[ 1.3939,  2.0481,  1.6241,  ...,  1.6422,  2.0329,  1.6559],
         [ 2.7510,  1.8494,  2.7831,  ...,  2.2829,  2.8543,  2.5999],
         [-0.2536, -0.2428, -0.2509,  ..., -0.1566,  0.3800, -0.3086],
         ...,
         [ 0.4430, -0.6355, -0.9793,  ...,  0.2358, -1.3083, -0.7309],
         [-1.3735, -1.7448, -2.0035,  ..., -2.0985, -1.9956, -1.9261],
         [ 1.7020,  1.4654,  2.3191,  ...,  2.3501,  2.0048,  1.6740]]]), hidden_states=None, attentions=None)

In [ ]:
from IPython.display import Audio

Audio(speech, rate=model.config.sampling_rate)


In [ ]:
speech.min()*32767

tensor(-21274.1758)

In [ ]:
target_dtype = np.int16
max_range = np.iinfo(target_dtype).max
max_range

32767

In [ ]:
#self._file.write(struct.pack('<L4s4sLHHLLHH4s',
#struct.error: ushort format requires 0 <= number <= (0x7fff * 2 + 1)
(0x7fff * 2 + 1)

65535